# Large-scale demo — t5-base ASTE on Yelp Restaurant Reviews (Tuần 5)

Runs the fine-tuned **t5-base** ASTE model on real, unlabeled restaurant reviews to demo the
pipeline at scale — the Tuần 5 "test report generation on a large dataset" requirement.

**Why Yelp Restaurant Reviews, not Amazon Reviews** (the originally-planned dataset in
`docs/Proposal.md`): the ASTE model is fine-tuned on SemEval Restaurant Triplet data — its
vocabulary of aspects/opinions is restaurant-specific (`food`, `service`, `staff`, `atmosphere`,
...). Amazon Reviews (planned as the Electronics category) is a different domain entirely and
would likely produce meaningless extractions. [Yelp Restaurant
Reviews](https://www.kaggle.com/datasets/farukalam/yelp-restaurant-reviews) is the same domain as
training data, so results should actually make sense.

Like the originally-planned Amazon Reviews demo, **this has no gold labels — it's a scale demo,
not a benchmark** (see `docs/Proposal.md`'s note on the role of the secondary dataset). Only a
`predicted` table is produced; there's no `gold` table to sanity-check against here, but the same
pipeline already validated at 97.25% majority-sentiment agreement on labeled SemEval Restaurant
data (`notebooks/aste_aspect_reasons_restaurant.ipynb`), which is some indirect evidence of
trustworthiness on this same-domain (if different-source) data.

**Design note — sentence splitting**: the model was fine-tuned on single, short SemEval sentences
(`MAX_INPUT_LENGTH=160` tokens). Real Yelp reviews are full multi-sentence paragraphs, so each
review is split into individual sentences before inference — feeding a whole paragraph directly
would truncate and likely produce poor/incomplete triplets.

**Kaggle setup**: add `farukalam/yelp-restaurant-reviews` and the t5-base ASTE model
(`t5-base-aste-restaurant-best/`) as inputs, GPU accelerator.

In [ ]:
import glob
import json
import re
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


## 1. Config

`SAMPLE_SIZE` caps how many **reviews** (not sentences — each review becomes several sentences
after splitting) are processed, so a first run stays fast; raise it for a bigger demo once this
runs cleanly. `SEED` matches the seed used everywhere else in this project for reproducibility.

In [ ]:
SAMPLE_SIZE = 2000  # number of reviews to sample; raise once this runs cleanly
SEED = 42
MAX_LENGTH = 160
BATCH_SIZE = 32
MIN_MENTIONS = 2
TOP_N_REASONS = 10

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")


## 2. Locate the Yelp CSV and the fine-tuned model

Searches `/kaggle/input` for a CSV containing a review-text-like column (tries the exact
`Review text` column name the dataset is expected to use, then falls back to any
case-insensitively-matching column, so this survives minor naming differences without editing
code). `find_model_dir` matches `aste-restaurant-best` — same lesson from earlier in this project
about Kaggle's "New Model from notebook output" also pulling in unrelated checkpoint dirs.

In [ ]:
def find_review_csv():
    candidates = list(INPUT_ROOT.glob("**/*.csv"))
    if not candidates:
        raise FileNotFoundError(
            "No CSV found under /kaggle/input. Add the 'farukalam/yelp-restaurant-reviews' "
            "dataset as this notebook's input."
        )
    # Prefer a CSV whose path mentions "yelp" if there are multiple candidates.
    yelp_candidates = [p for p in candidates if "yelp" in str(p).lower()]
    return sorted(yelp_candidates or candidates)[0]


def find_review_text_column(df):
    if "Review text" in df.columns:
        return "Review text"
    matches = [c for c in df.columns if "review" in c.lower() and "text" in c.lower()]
    if not matches:
        matches = [c for c in df.columns if "review" in c.lower()]
    if not matches:
        raise ValueError(
            f"Could not find a review-text column. Available columns: {list(df.columns)}. "
            "Set REVIEW_COLUMN manually below."
        )
    return matches[0]


CSV_PATH = find_review_csv()
print("Using CSV:", CSV_PATH)

reviews_df = pd.read_csv(CSV_PATH)
REVIEW_COLUMN = find_review_text_column(reviews_df)
print(f"Review text column: {REVIEW_COLUMN!r} | Columns: {list(reviews_df.columns)}")
print(f"Total reviews in file: {len(reviews_df)}")

reviews_df = reviews_df[[REVIEW_COLUMN]].dropna()
reviews_df = reviews_df.sample(n=min(SAMPLE_SIZE, len(reviews_df)), random_state=SEED).reset_index(drop=True)
print(f"Sampled {len(reviews_df)} reviews (SAMPLE_SIZE={SAMPLE_SIZE}, seed={SEED})")


def find_model_dir():
    candidates = []
    for config_path in glob.glob(f"{INPUT_ROOT}/**/config.json", recursive=True):
        d = Path(config_path).parent
        if (d / "model.safetensors").exists() or (d / "pytorch_model.bin").exists():
            candidates.append(d)
    candidates = sorted(set(candidates), key=str)
    final_candidates = [d for d in candidates if "aste-restaurant-best" in str(d).lower()]
    chosen = final_candidates or candidates
    if not chosen:
        raise FileNotFoundError(
            "No fine-tuned model found under /kaggle/input. Upload the t5-base-aste-restaurant-best/ "
            "folder saved by notebooks/train-t5-base-for-aste-on-14res-15res-16res.ipynb as a new "
            "Kaggle Dataset/Model and add it as this notebook's input."
        )
    if len(candidates) > 1:
        print(f"Found {len(candidates)} candidate model dir(s): {candidates}")
    return chosen[0]


MODEL_DIR = find_model_dir()
print("Using model dir:", MODEL_DIR)


## 3. Split reviews into sentences

Simple regex-based sentence splitter (splits on `.`/`!`/`?` followed by whitespace) — this
dataset is a scale demo, not a benchmark, so a lightweight heuristic is proportionate; no new
dependency (spaCy/NLTK) needed for this one-off split.

In [ ]:
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def split_sentences(text):
    text = str(text).strip()
    if not text:
        return []
    sentences = [s.strip() for s in SENTENCE_SPLIT_RE.split(text) if s.strip()]
    # Drop fragments with no real content (stray punctuation, single words) — keep short-but-valid
    # opinionated sentences like "Great atmosphere." or "Loved it!", which are common and exactly
    # the terse style the model was trained on.
    return [s for s in sentences if len(s.split()) >= 2 and re.search(r"[a-zA-Z]", s)]


all_sentences = []
for review in reviews_df[REVIEW_COLUMN]:
    all_sentences.extend(split_sentences(review))

print(f"{len(reviews_df)} reviews -> {len(all_sentences)} sentences ({len(all_sentences) / len(reviews_df):.1f} sentences/review)")


## 4. Run t5-base inference

Same batched beam-search generation as `notebooks/aste_aspect_reasons_restaurant.ipynb`.

In [ ]:
PREFIX = "extract aspect sentiment triplets: "
TRIPLET_RE = re.compile(
    r"aspect:\s*(.*?)\s*\|\s*opinion:\s*(.*?)\s*\|\s*sentiment:\s*(positive|negative|neutral)",
    re.IGNORECASE,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()
print("Device:", device)


def parse_predicted_triplets(text):
    if text.strip().lower() == "no triplet":
        return []
    return [
        (aspect.strip(), opinion.strip(), sentiment.strip().lower())
        for aspect, opinion, sentiment in TRIPLET_RE.findall(text)
    ]


predicted_texts = []
with torch.no_grad():
    for start in range(0, len(all_sentences), BATCH_SIZE):
        batch = [PREFIX + s for s in all_sentences[start : start + BATCH_SIZE]]
        inputs = tokenizer(batch, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
        output_ids = model.generate(**inputs, max_length=MAX_LENGTH, num_beams=4, early_stopping=True)
        predicted_texts.extend(tokenizer.batch_decode(output_ids, skip_special_tokens=True))
        done = min(start + BATCH_SIZE, len(all_sentences))
        print(f"  {done}/{len(all_sentences)}", end="\r")
print()

predicted_records = [triplet for text in predicted_texts for triplet in parse_predicted_triplets(text)]
print(f"Extracted {len(predicted_records)} triplets from {len(all_sentences)} sentences")


## 5. Aggregate: per-aspect counts + top-10 reasons per sentiment

Same `aggregate_aspect_reasons` logic as `src/report/aspect_stats.py` (tested locally in
`tests/test_aspect_stats.py`), inlined here.

In [ ]:
SENTIMENT_LABELS = ("positive", "negative", "neutral")


@dataclass
class AspectReasonSummary:
    aspect: str
    positive: int
    positive_reasons: list
    negative: int
    negative_reasons: list
    neutral: int
    neutral_reasons: list
    total: int
    majority_sentiment: str


def _top_reasons(counter, top_n):
    ranked = sorted(counter.items(), key=lambda item: (-item[1], item[0]))
    return ranked[:top_n]


def aggregate_aspect_reasons(records, top_n=10, min_mentions=1):
    counts = {}
    reason_counts = {}
    for aspect, opinion, sentiment in records:
        key = aspect.strip().lower()
        if not key or sentiment not in SENTIMENT_LABELS:
            continue
        counts.setdefault(key, Counter())[sentiment] += 1
        reason = opinion.strip().lower()
        if reason:
            reason_counts.setdefault(key, {}).setdefault(sentiment, Counter())[reason] += 1

    summaries = []
    for aspect, counter in counts.items():
        total = sum(counter.values())
        if total < min_mentions:
            continue
        majority_sentiment = max(
            SENTIMENT_LABELS, key=lambda label: (counter[label], -SENTIMENT_LABELS.index(label))
        )
        aspect_reasons = reason_counts.get(aspect, {})
        summaries.append(
            AspectReasonSummary(
                aspect=aspect,
                positive=counter["positive"],
                positive_reasons=_top_reasons(aspect_reasons.get("positive", Counter()), top_n),
                negative=counter["negative"],
                negative_reasons=_top_reasons(aspect_reasons.get("negative", Counter()), top_n),
                neutral=counter["neutral"],
                neutral_reasons=_top_reasons(aspect_reasons.get("neutral", Counter()), top_n),
                total=total,
                majority_sentiment=majority_sentiment,
            )
        )

    summaries.sort(key=lambda s: (-s.total, s.aspect))
    return summaries


predicted_summary = aggregate_aspect_reasons(predicted_records, top_n=TOP_N_REASONS, min_mentions=MIN_MENTIONS)

print(f"Aspects with >= {MIN_MENTIONS} mentions: {len(predicted_summary)}")
print()
print("Top 15 aspects by mentions:")
for s in predicted_summary[:15]:
    print(f"  {s.aspect:<20} total={s.total:4d}  +{s.positive:<4d} -{s.negative:<4d} ~{s.neutral:<4d}  majority={s.majority_sentiment}")


## 6. Save results

In [ ]:
def to_records(summary):
    return [
        {
            "aspect": s.aspect,
            "positive": s.positive,
            "positive_reasons": s.positive_reasons,
            "negative": s.negative,
            "negative_reasons": s.negative_reasons,
            "neutral": s.neutral,
            "neutral_reasons": s.neutral_reasons,
            "total": s.total,
            "majority_sentiment": s.majority_sentiment,
        }
        for s in summary
    ]


out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("output")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "aspect_reasons_yelp_demo.json"
out_path.write_text(json.dumps({
    "model_dir": str(MODEL_DIR),
    "source_csv": str(CSV_PATH),
    "domain": "restaurant (Yelp, unlabeled large-scale demo)",
    "sample_size_reviews": len(reviews_df),
    "num_sentences": len(all_sentences),
    "min_mentions": MIN_MENTIONS,
    "top_n_reasons": TOP_N_REASONS,
    "seed": SEED,
    "predicted": to_records(predicted_summary),
}, indent=2))
print(f"Saved aspect + reasons table to {out_path}")


## Next steps

- Hand off `aspect_reasons_yelp_demo.json` as the Tuần 5 "large-scale demo" input for report
  generation, alongside (not replacing) `output/aspect_reasons_restaurant.json` (the labeled
  SemEval Restaurant table used for the pipeline's own validation).
- If the demo needs to be bigger, raise `SAMPLE_SIZE` in Section 1 and re-run — everything else
  is unchanged.
- No gold labels here, so there's no majority-sentiment-agreement sanity check to run (unlike
  `aste_aspect_reasons_restaurant.ipynb`) — trustworthiness rests on that notebook's 97.25%
  agreement on labeled same-domain data.